In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import json
import re
import gc
import subprocess
import unicodedata
import torch
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

modelo_aluno_nome = "Qwen/Qwen2.5-7B-Instruct"
modelo_professor_chave = "qwen/qwen-2.5-72b-instruct"

caminho_labels_treino = "/home/cecilia/Documentos/PIBIC/Rotulagem_Finetuning/labels_treino .json"
caminho_labels_teste  = "/home/cecilia/Documentos/PIBIC/Rotulagem_Finetuning/labels_teste .json"
caminho_modelo_final  = "./aluno_fine_tunado"

In [ ]:
#Script para descartar um treino mal executado (opcional)
#import shutil

#if os.path.isdir(caminho_modelo_final):
    #shutil.rmtree(caminho_modelo_final)
    #print("Checkpoints antigos removidos.")

In [ ]:
with open(caminho_labels_treino, 'r', encoding='utf-8') as f:
    labels_treino = json.load(f)[modelo_professor_chave]

with open(caminho_labels_teste, 'r', encoding='utf-8') as f:
    labels_teste = json.load(f)[modelo_professor_chave]

print(f"Treino: {len(labels_treino)} reclamações | Teste: {len(labels_teste)} reclamações")

## Carregamento do modelo aluno com LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(modelo_aluno_nome)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

aluno = AutoModelForCausalLM.from_pretrained(
    modelo_aluno_nome,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

aluno = get_peft_model(aluno, lora_config)
aluno.enable_input_require_grads()
aluno.print_trainable_parameters()

In [7]:
system_prompt_aluno = """
Persona:
Você é um especialista em análise de reclamações online, especificamente no CRM, responsável por desenvolver estratégias de negócios, com foco em entender as necessidades do consumidor.

Contexto:
    - O dataset contém reclamações extensas de consumidores online sobre diversos domínios
    - Cada reclamação contém pelo menos um alvo principal específico (podem ter mais de um), correspondente à frase específica a qual o cliente expressa o problema mais relevante da reclamação.
    - Termo de aspecto: Atributo específico ao qual a frase se refere.
    - Categoria do aspecto: Par no formato Entidade#Atributo, onde o 1º refere-se ao elemento da reclamação, e o 2º representa a dimensão que esta sendo avaliada.

Tarefa:
Você deve extrair de cada reclamação, um par contendo o aspecto e sua categoria.
Para isso você deve seguir os passos:
1- Encontre dentro da reclamação o alvo principal, frase a qual contém o problema central da reclamação. (podendo haver mais de um alvo)
2- Dentro do alvo (para cada alvo), encontre um par contendo o termo de aspecto e a sua respectiva categoria
3- Retorne na saída o alvo encontrado e embaixo o par solicitado.

Siga o modelo abaixo para a saída:
	Alvo principal: o alvo principal da reclamação,
	Rótulo (s): aspecto|CATEGORIA

#########Atenção######:

Quando existir mais de um alvo por reclamação, retorne cada um com seu respectivo par abaixo, como no modelo a seguir:
        Alvo 1: "1º alvo que você encontrar"
        Rótulo(s): aspecto|CATEGORIA

        Alvo 2: "2º alvo que você encontrar"
        Rótulo(s): aspecto|CATEGORIA

Quando existir mais de um aspecto e categoria para o mesmo alvo, separe cada par por ';'. Siga este modelo:
	Rótulo(s): aspecto|CATEGORIA; aspecto|CATEGORIA

Quando o aspecto não estiver explicitamente escrito no texto, infera-o pelo contexto e escreva o Aspecto inferido e o termo 'implícito' entre '()'. Como no exemplo a seguir:
	Rótulo(s): Aspecto (implícito)|CATEGORIA

Se não for possível inferir nenhum aspecto, retorne somente a palavra 'implícito' no lugar do aspecto e a sua respectiva categoria:
	Rótulo(s): Implícito|CATEGORIA

#######Observações########:
- Separe cada par por uma barra vertical como esta: '|'
- Não extraia aspectos mencionados apenas como contexto, histórico ou consequência do problema principal. Priorize sempre o aspecto diretamente associado à reclamação central do consumidor.
- Não extraia aspectos secundários ou periféricos.
- Retorne somente as categorias em caixa alta. O termo de aspecto deve manter a capitalização natural do texto.
- Não utilize aspas (retas, curvas ou de qualquer tipo) ao redor do alvo ou do texto extraído. Escreva o texto diretamente, sem aspas envolvendo.

##########EXEMPLOS##############

######Exemplo1########

Reclamação completa:
"Boa tarde Empréstimo consignado já foi pago documentos e contrato não consiste com vício documentos peço o cancelamento do contrato do banco inter"

Alvo principal: Empréstimo consignado já foi pago
Rótulo(s): contrato|CONTRATO#CANCELAMENTO

######Exemplo2########

Reclamação completa:
"Após o recebimento de ligação de cobrança, fui induzido a ingressar na plataforma do SERASA LIMPA NOME, na qual pude constatar a existência de um débito em meu nome no valor de R$ 3.084,11, correspondente ao contrato nº 1662873 débito este não reconhecido pelo NOTIFICANTE."

Alvo 1: constatar a existência de um débito em meu nome
Rótulo(s): débito|COBRANÇA#INDEVIDA

Alvo 2: débito este não reconhecido pelo NOTIFICANTE
Rótulo(s): débito|COBRANÇA#RECONHECIMENTO

######Exemplo3########

Reclamação completa:
"Boa sorte para conseguir uma mesa."

Alvo principal: Boa sorte para conseguir uma mesa (implícito)
Rótulo(s): Implícito|RESTAURANTE#DIVERSOS

######Exemplo4########

Reclamação completa:
"Ansioso por mais leituras dos mesmos autores."

Alvo 1: leituras
Rótulo(s): leituras|LIVRO#GERAL

Alvo 2: autores
Rótulo(s): autores|LIVRO#AUTOR

######Exemplo5########

Reclamação completa:
"Comprei uma passagem para o voo AZUL2233 que deveria decolar às 14h, mas fomos informados apenas 20 minutos antes do horário previsto que o voo seria cancelado, sem nenhuma alternativa oferecida pela companhia até o momento."

Alvo principal: fomos informados apenas 20 minutos antes do horário previsto que o voo seria cancelado, sem nenhuma alternativa oferecida pela companhia até o momento

Rótulo(s): cancelamento|VOO#CANCELAMENTO; informação|ATENDIMENTO#COMUNICAÇÃO
"""

## Funções de prompt, parsing, extração e métricas

In [8]:
def montar_prompt(texto_cliente):
    return [
        {"role": "system", "content": system_prompt_aluno},
        {"role": "user", "content": f"Input: {texto_cliente}\nOutput:"}
    ]

def limpar_aspas(texto):
    if texto is None:
        return texto
    return texto.strip().strip('"').strip("'").strip()

def parse_output(texto):
    pares = []
    blocos = re.split(r'(?=Alvo\s*(?:principal)?\s*\d*\s*[:"])', texto, flags=re.IGNORECASE)
    for bloco in blocos:
        bloco = bloco.strip()
        if not bloco:
            continue
        alvo_match = re.search(r'Alvo[^:]*:\s*"?([^"\n]+)"?', bloco, flags=re.IGNORECASE)
        alvo = limpar_aspas(alvo_match.group(1)) if alvo_match else None
        rotulo_match = re.search(r'R[oó]tulo\(?s?\)?:?\s*"?(.+)', bloco, flags=re.IGNORECASE | re.DOTALL)
        rotulo = limpar_aspas(rotulo_match.group(1)) if rotulo_match else None
        if alvo and rotulo:
            pares.append((alvo, rotulo))
    return pares

def normalizar(texto):
    texto = str(texto).strip().lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

@torch.no_grad()
def extrair_com_aluno(modelo, tokenizer, dados, max_novos_tokens=800, caminho_checkpoint=None):
    resultados = []
    modelo.eval()

    indices_ja_feitos = set()
    if caminho_checkpoint:
        try:
            with open(caminho_checkpoint, 'r', encoding='utf-8') as f:
                resultados = json.load(f)
                indices_ja_feitos = set(item["index"] for item in resultados)
                print(f"Checkpoint carregado: {len(resultados)} já processados.")
        except FileNotFoundError:
            pass

    for item in tqdm(dados, desc="Extraindo"):
        if item["index"] in indices_ja_feitos:
            continue

        mensagens = montar_prompt(item["input"]["texto_cliente"])
        entrada = tokenizer.apply_chat_template(
            mensagens, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True
        ).to(modelo.device)

        saida_ids = modelo.generate(
            **entrada, max_new_tokens=max_novos_tokens, do_sample=False
        )
        texto_saida = tokenizer.decode(
            saida_ids[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True
        )

        resultados.append({
            "index": item["index"],
            "previsto": parse_output(texto_saida),
            "esperado": parse_output(item["output"])
        })

        if caminho_checkpoint:
            with open(caminho_checkpoint, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)

    return resultados

def calcular_metricas(resultados):
    vp = fp = fn = uniao_total = 0
    for item in resultados:
        previstos = set((normalizar(a), normalizar(r)) for a, r in item["previsto"])
        esperados = set((normalizar(a), normalizar(r)) for a, r in item["esperado"])
        acertos = previstos & esperados
        uniao = previstos | esperados
        vp += len(acertos)
        fp += len(previstos - esperados)
        fn += len(esperados - previstos)
        uniao_total += len(uniao)

    precisao = vp / (vp + fp) if (vp + fp) > 0 else 0
    recall   = vp / (vp + fn) if (vp + fn) > 0 else 0
    f1       = 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0
    acuracia = vp / uniao_total if uniao_total > 0 else 0

    return {"precisao": precisao, "recall": recall, "f1": f1, "acuracia": acuracia}

## Script para avaliar o aluno antes do fine-tuning

In [ ]:
print("Avaliando o aluno sem fine-tuning...")
resultados_baseline = extrair_com_aluno(
    aluno, tokenizer, labels_teste,
    caminho_checkpoint="resultados_baseline_detalhado.json"
)
metricas_baseline = calcular_metricas(resultados_baseline)
print("Métricas baseline:", metricas_baseline)

with open("metricas_baseline.json", "w", encoding="utf-8") as f:
    json.dump(metricas_baseline, f, ensure_ascii=False, indent=4)

## Limpeza de memória da GPU antes do treinamento 


In [ ]:
torch.cuda.empty_cache() 
gc.collect()

## Tokenização dos dados de treino

In [ ]:
def tokenizar_exemplo(item, max_length=1600):
    mensagens = montar_prompt(item["input"]["texto_cliente"])
    prompt_texto = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    texto_completo = prompt_texto + item["output"] + tokenizer.eos_token

    codificado = tokenizer(
        texto_completo, truncation=True, max_length=max_length, padding="max_length"
    )

    prompt_ids = tokenizer(prompt_texto, truncation=True, max_length=max_length)["input_ids"]
    tamanho_prompt = len(prompt_ids)

    labels = codificado["input_ids"].copy()
    for i in range(min(tamanho_prompt, len(labels))):
        labels[i] = -100

    # mascarar também o padding, usando a attention_mask
    for i, mask in enumerate(codificado["attention_mask"]):
        if mask == 0:
            labels[i] = -100

    codificado["labels"] = labels
    return codificado

dataset_treino = Dataset.from_list(
    [tokenizar_exemplo(item) for item in tqdm(labels_treino, desc="Tokenizando")]
)

## Configuração do treino (ajuste dos hiperparâmetros)

In [11]:
training_args = TrainingArguments(
    output_dir=caminho_modelo_final,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    disable_tqdm=False,
    report_to="none"
)

## Monitoramento da GPU (uso de memória, utilização e temperatura da GPU)

In [12]:
class MonitorGPUCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            resultado = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
                 "--format=csv,noheader"],
                capture_output=True, text=True
            )
            print(f"[Passo {state.global_step}] GPU: {resultado.stdout.strip()}")

## Execução do treino

In [ ]:
trainer = Trainer(
    model=aluno,
    args=training_args,
    train_dataset=dataset_treino,
    callbacks=[MonitorGPUCallback()]
)

ultimo_checkpoint = None
if os.path.isdir(caminho_modelo_final):
    checkpoints = [d for d in os.listdir(caminho_modelo_final) if d.startswith("checkpoint-")]
    if checkpoints:
        ultimo_checkpoint = os.path.join(
            caminho_modelo_final,
            sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        )
        print(f"Retomando do checkpoint: {ultimo_checkpoint}")

trainer.train(resume_from_checkpoint=ultimo_checkpoint)
aluno.save_pretrained(caminho_modelo_final)
tokenizer.save_pretrained(caminho_modelo_final)

## Avaliação do modelo aluno após o fine-tuning

In [ ]:
print("Avaliando o aluno com fine-tuning...")
resultados_pos_ft = extrair_com_aluno(
    aluno, tokenizer, labels_teste,
    caminho_checkpoint="resultados_pos_ft_detalhado.json"
)
metricas_pos_ft = calcular_metricas(resultados_pos_ft)
print("Métricas pós-fine-tuning:", metricas_pos_ft)

with open("metricas_pos_ft.json", "w", encoding="utf-8") as f:
    json.dump(metricas_pos_ft, f, ensure_ascii=False, indent=4)

In [ ]:
import json

with open("metricas_baseline.json", "r", encoding="utf-8") as f:
    metricas_baseline = json.load(f)

print("Métricas baseline recarregadas:", metricas_baseline)

## Comparação das métricas do modelo aluno antes e depois do fine-tuning

In [ ]:
print("\n=== COMPARAÇÃO ===")
for chave in metricas_baseline:
    antes = metricas_baseline[chave]
    depois = metricas_pos_ft[chave]
    print(f"{chave}: {antes:.3f} -> {depois:.3f} (Δ {depois - antes:+.3f})")